# Fourier Feature Network 2D Image Regression
## Configurable version of the architecture for 2D Image Regression tasks, with UI.

This notebook provides a user interface that allows to configure most of the model's parameters and the training process. "Dementia Factor" parameter is introduced for the evaluation stage, which allows the user to controll the global dropout across the model's weights.

## Instructions
If you wish to run this notebook, please follow the instructions below. For a more detailed explanation of the architecture implementation, please refer to the _Github ReadMe_ file.

To run this notebook:
1. First run the cell below (Base Functions).
2. Then, run the next cell (UI).
3. Wait for the UI to fully load inside the colab cell.
4. **Do not close** the colab browser tab or stop the cell until you are ready to finish. It is also recommended to open the colab tab periodically if you let it run in the backgound. Consider saving your models after training, colab may disconnect you and wipe the progress at any moment.

_Alternatively you can click the Gradio link above the UI window to open it in a separate window. The console output, which might be useful, will still be visible in colab below the UI window, although contents of the window will not be carried over._

##### After you have set up the environment, you can choose to either **Load** or **Train** the model, upload the **Training Image** if relevant, and configure the training and evaluation **Parameters**.

In the bottom of the User interface, you will find the **Output** section, which will display the output of the chosen operation, as well a _Status Window_, which will display any errors that might have occured. Also, you will notice the _Save Active Model_ and _Reload_ buttons to the side of the _Status Window_, which allow you to save the current trained or loaded model, or re-evaluate the model that is currently in memory. You can find the saved model file in in the "files" tab to the left in Colab. You might have to look for the folder that the environment uses, usually "content".

##### The **Evaluation Parameters** section allows you to controll the model's output, generated during the evaluation stage.

- The so-called _"Dementia Factor"_ is a percentage, which controls how much of the model's weights are zeroed-out before generating output. Note that it does not damage the weights permanently, each evaluation run is performed on a safe copy.

- _Super Resolution_ consists of two fields, _SR Height_ and _SR Width_, they allow you to perform Super Resolution on the image and let the model generate an upscaled or downscaled version of the input with the specified resoltion.
_You may keep the fields at 0, 0 if you wish the output to retain the original resolution and aspect ratio._

##### **Training Parameters** allow you to configure the training process for the model.

**Basic Parameters**:
- _Epochs_ defines how many iterations will the training run for. Single iteration processes each pixel in the input image once. Depending on the nature of the image, it may take significantly more or less iterations to reach acceptable fidelity. For most "nature" scenes 10 - 20 epochs is sufficient for decent fidelity, below 10 will result in decline of quality, but 1 - 5 range might produce interesting results. _Note that training takes a long time on colab, 10 epochs might train for 10 - 20 minutes. Consider switching to local environments for a **significant** speedup._
- _Scale_ parameter is one of the major factors that impact the capacity of the model to learn complex imagery. Values between 8 - 20 are recommended. Going over would result in more noise, going under will cause the model to degrade in it's abilities.
- SR and "Dementia Factor" are as in **Evaluation Parameters**. Will only affect the first output of the model after training.

For **Advanced Parameters**, please refer to the _Github ReadMe_ file.

## Notes
Currently you cannot continue training of an already trained model. Clicking train simply starts training of a completely new model, regardless if there is an already loaded model present in memory.

Currently it is not possible to save the model during training.

It is recommended to run the training locally if possible, since colab slows down training by 5 - 8 times compared to mid-high range consumer grade GPUs. This notebook was not tested locally, but it may still run. You may run the clean (no UI) version of the script, if the latter does not work.

Cell: Base Functions

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.optim import Adam
from PIL import Image
from torchvision.utils import save_image
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
import os
import copy




class FourierFeatureMapping(nn.Module):
    def __init__(self, input_dim, output_dim, scale, mapping):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.scale = scale
        self.mapping = mapping
        B_matrix = None

        if self.mapping == "gaussian":
            B_matrix = torch.normal(mean=0, std=scale, size=(output_dim, input_dim))
            self.output_dim = output_dim * 2
        elif self.mapping == "basic":
            B_matrix = torch.eye(input_dim)
            self.output_dim = input_dim * 2
        else:
            raise ValueError(f"Mapping \"{self.mapping}\" is not implemented.\nPlease check the documentation for the implemented mappings.")

        self.register_buffer('B', B_matrix)

    def forward(self, x):
        theta = 2 * torch.pi * (x @ self.B.t())     # order reversed since input will be batched. (N, d) @ (m, d)^T = (N, m)
        return torch.cat([torch.cos(theta), torch.sin(theta)], dim=-1)


class FourierFeatureNetwork(nn.Module):
    def __init__(self, input_dim, output_dim, scale=10, mapping="gaussian", mapping_dim=256, layers=4, hidden_dim=256):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.layers = layers
        self.hidden_dim = hidden_dim

        self.ffm = FourierFeatureMapping(input_dim, mapping_dim, scale, mapping)
        mlp_layers = []
        layer_in_dim = self.ffm.output_dim
        for _ in range(self.layers):
            mlp_layers.append(nn.Linear(layer_in_dim, hidden_dim))
            mlp_layers.append(nn.ReLU())
            layer_in_dim = hidden_dim

        mlp_layers.append(nn.Linear(layer_in_dim, output_dim))
        mlp_layers.append(nn.Sigmoid())
        self.mlp = nn.Sequential(*mlp_layers)

    def forward(self, x):
        x = self.ffm(x)
        return self.mlp(x)


class ImageCoordinateDataset(Dataset):
    def __init__(self, pth, size=(0, 0)):
        super().__init__()

        img = Image.open(pth).convert('RGB')
        if size != (0,0):
            img = img.resize(size)
        self.img_tensor = transforms.ToTensor()(img)
        _, self.H, self.W = self.img_tensor.shape

    def __len__(self):
        return self.H * self.W

    def get_h(self):
        return self.H

    def get_w(self):
        return self.W

    def __getitem__(self, idx):
        row = idx // self.W
        col = idx % self.W

        x = col / (self.W - 1) if self.W > 1.0 else 0.0
        y = row / (self.H - 1) if self.H > 1.0 else 0.0

        coords = torch.tensor([x, y], dtype=torch.float32)
        rgb = self.img_tensor[:, row, col]

        return coords, rgb


def train_ffn(epochs,
              img_pth,
              device,
              img_size=(0, 0),
              batch_size=65536,
              input_dim=2,
              output_dim=3,
              scale=10,
              mapping="gaussian",
              mapping_dim=256,
              hidden_dim=256,
              layers=4,
              num_workers=0):
    dataset = ImageCoordinateDataset(img_pth, img_size)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)

    model = FourierFeatureNetwork(input_dim, output_dim, scale, mapping=mapping, mapping_dim=mapping_dim, layers=layers, hidden_dim=hidden_dim).to(device)
    criterion = nn.MSELoss()
    optimizer = Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.999), eps=1e-08)

    for epoch in range(epochs):
        epoch_loss = 0.0
        for coords, rgb in dataloader:
            coords, rgb = coords.to(device), rgb.to(device)

            optimizer.zero_grad()
            loss = criterion(model(coords), rgb)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        print(f"Epoch: {epoch+1}/{epochs} | Loss: {epoch_loss / len(dataloader):.6f}")

    params = {'h': dataset.get_h(),
              'w': dataset.get_w(),
              'input_dim': input_dim,
              'output_dim': output_dim,
              'scale': scale,
              'mapping': mapping,
              'mapping_dim': mapping_dim,
              'hidden_dim': hidden_dim,
              'layers': layers}
    return model, params


def save_model(state_dict, params):
    now = datetime.now()
    t_now = now.strftime("%Y_%m_%d_%H_%M")

    os.makedirs("models", exist_ok=True)
    savefile = {'state_dict': state_dict,
                'param_dict': params}
    torch.save(savefile, f'models/FFN2dImReg_{t_now}.pth')


def eval_model(model, device, h_target, w_target, batch_size=8192, dementia_factor=0.0):
    model = copy.deepcopy(model)
    model.eval()

    x_coords = torch.linspace(0.0, 1.0, w_target)
    y_coords = torch.linspace(0.0, 1.0, h_target)
    grid_y, grid_x = torch.meshgrid(y_coords, x_coords, indexing='ij')
    coords = torch.stack([grid_x.flatten(), grid_y.flatten()], dim=-1)

    from torch.utils.data import TensorDataset
    dataset = TensorDataset(coords)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    predictions = []
    with torch.no_grad():
        if dementia_factor > 0.0:
            layer_id = 0
            for l in model.mlp:
                if layer_id % 2 == 0:
                    mask = torch.rand_like(model.mlp[layer_id].weight) < dementia_factor
                    model.mlp[layer_id].weight[mask] = 0

                layer_id += 1

        for (batch_coords, ) in dataloader:
            batch_coords = batch_coords.to(device)
            preds = model(batch_coords)
            predictions.append(preds)

    recreated_tensor = torch.cat(predictions, dim=0).view(h_target, w_target, 3).detach().cpu()
    return recreated_tensor


def save_eval_img(img_tensor):
    chw_tensor = img_tensor.permute(2, 0, 1)
    now = datetime.now()
    t_now = now.strftime("%Y_%m_%d_%H_%M")

    os.makedirs("saved_images", exist_ok=True)
    save_image(chw_tensor, f'saved_images/{t_now + "_" + str(np.random.randint(0, 19))}.png')


def plot_eval_img(img_tensor):
    img_np = img_tensor.numpy()
    h, w, _ = img_np.shape

    fig_width = 8
    fig_height = fig_width * (h/w)
    fig = plt.figure(figsize=(fig_width, fig_height))
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis('off')
    ax.imshow(img_np, aspect='auto')
    plt.show()

Cell: UI

In [4]:
import gradio as gr
import torch
import torchvision.transforms as T
from PIL import Image




def train_and_eval(
    input_image, epochs, batch_size, scale, mapping,
    mapping_dim, hidden_dim, layers, sr_h, sr_w, dementia_factor
):
    if input_image is None:
        return None, "Could not start training. Please upload a training image.", None, None

    temp_img_pth = "temp_input.jpg"
    input_image.save(temp_img_pth)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


    model, param_dict = train_ffn(
        epochs=int(epochs),
        img_pth=temp_img_pth,
        device=device,
        batch_size=int(batch_size),
        scale=float(scale),
        mapping=mapping,
        mapping_dim=int(mapping_dim),
        hidden_dim=int(hidden_dim),
        layers=int(layers),
        num_workers=0
    )


    h = param_dict['h']
    w = param_dict['w']
    if int(sr_h) == 0 and int(sr_w) == 0:
        target_h = int(h)
        target_w = int(w)
    else:
        target_h = int(sr_h)
        target_w = int(sr_w)

    out_tensor = eval_model(model, device, target_h, target_w, dementia_factor=float(dementia_factor))
    out_tensor_chw = out_tensor.permute(2, 0, 1).clamp(0, 1)
    output_image = T.ToPILImage()(out_tensor_chw)

    return output_image, "Training complete.", model.state_dict(), param_dict


def load_and_eval(
    dementia_factor, sr_h, sr_w, model_file
):
    if model_file is None:
        return None, "Could not load the model. Please upload a .pth model file.", None, None


    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    savefile = torch.load(model_file)
    param_dict = savefile['param_dict']
    model = FourierFeatureNetwork(input_dim=param_dict['input_dim'], output_dim=param_dict['output_dim'], scale=param_dict['scale'], mapping=param_dict['mapping'], mapping_dim=param_dict['mapping_dim'], hidden_dim=param_dict['hidden_dim'], layers=param_dict['layers']).to(device)
    model.load_state_dict(savefile['state_dict'])
    h = param_dict['h']
    w = param_dict['w']
    if int(sr_h) == 0 and int(sr_w) == 0:
        target_h = int(h)
        target_w = int(w)
    else:
        target_h = int(sr_h)
        target_w = int(sr_w)

    out_tensor = eval_model(model, device, target_h, target_w, dementia_factor=float(dementia_factor))
    out_tensor_chw = out_tensor.permute(2, 0, 1).clamp(0, 1)
    output_image = T.ToPILImage()(out_tensor_chw)

    return output_image, "Model loaded.", model.state_dict(), param_dict


def eval_state(
    dementia_factor, sr_h, sr_w, state_dict, param_dict
):
    if state_dict is None or param_dict is None:
        return None, "Could not reload the model, no model exists in memory. Train or load one first."


    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = FourierFeatureNetwork(input_dim=param_dict['input_dim'], output_dim=param_dict['output_dim'], scale=param_dict['scale'], mapping=param_dict['mapping'], mapping_dim=param_dict['mapping_dim'], hidden_dim=param_dict['hidden_dim'], layers=param_dict['layers']).to(device)
    model.load_state_dict(state_dict)
    h = param_dict['h']
    w = param_dict['w']
    if int(sr_h) == 0 and int(sr_w) == 0:
        target_h = int(h)
        target_w = int(w)
    else:
        target_h = int(sr_h)
        target_w = int(sr_w)

    out_tensor = eval_model(model, device, target_h, target_w, dementia_factor=float(dementia_factor))
    out_tensor_chw = out_tensor.permute(2, 0, 1).clamp(0, 1)
    output_image = T.ToPILImage()(out_tensor_chw)

    return output_image, "Model reloaded."


def save_handler(state_dict, param_dict):
    if state_dict is None:
        return "Could not save the model, no model exists in memory. Train or load one first."

    save_model(state_dict, param_dict)
    return "Model saved to the 'models' directory."


# UI
hide_footer_css = """footer {display: none !important;}"""
with gr.Blocks(css=hide_footer_css, title="Fourier Feature Network 2D Image Regression") as ui:
    gr.Markdown("# Fourier Feature Network 2D Image Regression")


    with gr.Row():
        gr.Markdown("## Load or Train the model:")
    with gr.Row():
        gr.Markdown("#### Upload the Input Image _or_ the saved Model File _(.pth)_")

    with gr.Row():
        with gr.Column():
            input_image = gr.Image(type="pil", label="Input Image")

        with gr.Column():
            model_file = gr.File(label="Model Weights (.pth)", file_types=[".pth"], type="filepath")


    with gr.Row():
        gr.Markdown("## Training Parameters:")
    with gr.Row():
        gr.Markdown("#### _Not relevant for the loaded models. Can be ignored._")


    with gr.Row():
        with gr.Column():
            gr.Markdown("##### Basic Settings:")
            with gr.Row():
                epochs = gr.Textbox(value="10", label="Epochs:")
                scale = gr.Textbox(value="10.0", label="Scale:")
                dementia_factor_tr = gr.Textbox(value="0.0", label="Dementia Facor:")
            with gr.Row():
                gr.Markdown("##### Super Resolution:")
                gr.Markdown("###### _Keep at 0 to keep the input resolution._")
                sr_h = gr.Textbox(value="0", label="SR Height")
                sr_w = gr.Textbox(value="0", label="SR Width")


        with gr.Column():
            gr.Markdown("##### Advanced Settings:")
            mapping = gr.Radio(choices=["gaussian", "basic"], value="gaussian", label="Mapping Type:")
            mapping_dim = gr.Textbox(value="256", label="Mapping Dimension (Number of Frequencies)")
            hidden_dim = gr.Textbox(value="256", label="Hidden Dimension")
            layers = gr.Textbox(value="4", label="Number of MLP Layers")
            batch_size = gr.Dropdown(choices=[4096, 8192, 16384, 32768, 65536], value=65536, label="Batch Size")


    with gr.Row():
        gr.Markdown("## Evaluation Parameters:")
    with gr.Row():
        gr.Markdown("#### _Affects the loaded and current reloaded models. Does not influence the training._")

    with gr.Row():
        dementia_factor_ev = gr.Textbox(value="0.0", label="Dementia Facor:")
    with gr.Row():
        gr.Markdown("##### Super Resolution:")
        gr.Markdown("###### _Keep at 0 to keep the input resolution._")
        sr_h_ev = gr.Textbox(value="0", label="SR Height")
        sr_w_ev = gr.Textbox(value="0", label="SR Width")


    with gr.Row():
        gr.Markdown("## Start Training or Load:")

    with gr.Row():
        train_btn = gr.Button("Train", variant="primary")
        load_btn = gr.Button("Load")


    with gr.Row():
        gr.Markdown("## Output:")

    with gr.Row():
        output_image = gr.Image(type="pil", label="Reconstructed Image")
    with gr.Row():
        with gr.Column():
            status_box = gr.Textbox(label="System Status", interactive=False)
        with gr.Column():
            save_btn = gr.Button("Save Active Model")
            reload_btn = gr.Button("Reload", variant="primary")


    current_model_state_dict = gr.State()
    current_param_dict = gr.State()


    train_btn.click(
        fn=train_and_eval,
        inputs=[input_image, epochs, batch_size, scale, mapping, mapping_dim, hidden_dim, layers, sr_h, sr_w, dementia_factor_tr],
        outputs=[output_image, status_box, current_model_state_dict, current_param_dict]
    )

    load_btn.click(
        fn=load_and_eval,
        inputs=[dementia_factor_ev, sr_h_ev, sr_w_ev, model_file],
        outputs=[output_image, status_box, current_model_state_dict, current_param_dict]
    )

    reload_btn.click(
        fn=eval_state,
        inputs=[dementia_factor_ev, sr_h_ev, sr_w_ev, current_model_state_dict, current_param_dict],
        outputs=[output_image, status_box]
    )

    save_btn.click(
        fn=save_handler,
        inputs=[current_model_state_dict, current_param_dict],
        outputs=[status_box]
    )


ui.launch(debug=True)

/tmp/ipykernel_4857/2815076780.py:115: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=hide_footer_css, title="Fourier Feature Network 2D Image Regression") as ui:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://40a46fe80c9b9b50ae.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://40a46fe80c9b9b50ae.gradio.live
